In [2]:
!pip install langchain-text-splitters

  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 4.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.0/805.0 kB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.9/562.9 kB 1.3 MB/s  0:00:00 eta 0:00:01
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.6/640.6 kB 3.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [langchain-text-splitters]in-core]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [126]:
!pip install bert-score


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [127]:
import json
import re
from pathlib import Path

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_MD = "output2.md"
OUTPUT_JSON = "chunks.json"
CHUNK_OVERLAP = 153
CHUNK_SIZE = 1024 - CHUNK_OVERLAP
MIN_CHUNK_SIZE = 512 - CHUNK_OVERLAP
TOKENIZER = "cl100k_base"

RE_IMG = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")
RE_BLOCK_FORMULA = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)
RE_INLINE_FORMULA = re.compile(r"(?<!\$)\$([^\$\n]+?)\$(?!\$)")
RE_HTML_TABLE = re.compile(r"<table\b.*?</table>", re.DOTALL | re.I)
RE_MD_TABLE = re.compile(r"(?:^\|[^\n]*\|\s*$\n?){2,}", re.M)
PH_RE = re.compile(r"(\[\[(?:FORMULA|TABLE|IMAGE)_\d+\]\])")
RE_SENT = re.compile(r"(?<=[.!?])\s+")

enc = tiktoken.get_encoding(TOKENIZER)

FORMULAS = {}
TABLES = {}
IMAGES = {}
STORE = {}

def protect_pattern(text, pattern, store, prefix, block=False):
    def repl(m):
        key = f"[[{prefix}_{len(store):04d}]]"
        store[key] = m.group(0)
        return f"\n\n{key}\n\n" if block else key
    return pattern.sub(repl, text)


def protect_images(text):
    def repl(m):
        alt, src = m.group(1), m.group(2)
        key = f"[[IMAGE_{len(IMAGES):04d}]]"
        IMAGES[key] = {
            "alt": alt,
            "src": None if src.startswith("data:image") else src,
            "is_base64": src.startswith("data:image"),
            "src_len": len(src),
        }
        return key
    return RE_IMG.sub(repl, text)


def protect(text):
    text = protect_pattern(text, RE_BLOCK_FORMULA, FORMULAS, "FORMULA", block=True)
    text = protect_pattern(text, RE_HTML_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_MD_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_INLINE_FORMULA, FORMULAS, "FORMULA", block=False)
    return text


def restore(text):
    for k, v in STORE.items():
        if k in text:
            text = text.replace(k, v)
    return text


def count(text):
    total = 0
    for part in PH_RE.split(text):
        if not part:
            continue
        value = STORE.get(part)
        total += len(enc.encode(value if isinstance(value, str) else part))
    return total

def _tail_for_overlap(text, max_tokens):
    sentences = RE_SENT.split(text)
    picked, total = [], 0
    for s in reversed(sentences):
        if PH_RE.search(s):
            continue
        s_tokens = count(s)
        if s_tokens > max_tokens:
            continue
        if picked and total + s_tokens > max_tokens:
            break
        picked.insert(0, s)
        total += s_tokens
        if total >= max_tokens:
            break
    return " ".join(picked)

def add_overlap(chunks, overlap_tokens, chunk_size):
    if len(chunks) < 2:
        return chunks
    result = [chunks[0]]
    for i in range(1, len(chunks)):
        budget = chunk_size - count(chunks[i])
        budget = max(0, min(budget, overlap_tokens))
        tail = _tail_for_overlap(chunks[i-1], budget) if budget > 0 else ""
        result.append(f"{tail}\n\n{chunks[i]}" if tail else chunks[i])
    return result

def merge_small(chunks):
    if not chunks:
        return []
    merged, buf = [], ""
    for c in chunks:
        cand = f"{buf}\n\n{c}".strip() if buf else c
        if count(cand) <= CHUNK_SIZE:
            buf = cand
        else:
            if buf:
                merged.append(buf)
            buf = c
    if buf:
        merged.append(buf)

    i = 0
    while i < len(merged):
        if count(merged[i]) >= MIN_CHUNK_SIZE:
            i += 1
            continue
        done = False
        if i > 0 and count(merged[i - 1] + "\n\n" + merged[i]) <= CHUNK_SIZE:
            merged[i - 1] += "\n\n" + merged[i]
            merged.pop(i)
            done = True
        if not done and i < len(merged) - 1 and count(merged[i] + "\n\n" + merged[i + 1]) <= CHUNK_SIZE:
            merged[i + 1] = merged[i] + "\n\n" + merged[i + 1]
            merged.pop(i)
            done = True
        if not done:
            i += 1
    return merged


def chunk_markdown(text):
    FORMULAS.clear()
    TABLES.clear()
    IMAGES.clear()
    STORE.clear()

    text = protect_images(text)
    protected = protect(text)
    STORE.update(FORMULAS)
    STORE.update(TABLES)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=0,
        length_function=count,
        separators=["\n\n", "\n", " ", ""],
        keep_separator=True,
    )
    pieces = splitter.split_text(protected)
    pieces = merge_small(pieces)
    pieces = add_overlap(pieces, CHUNK_OVERLAP, CHUNK_SIZE)

    chunks = []
    for i, p in enumerate(pieces):
        restored = restore(p)
        img_refs = [{"placeholder": k, **IMAGES[k]} for k in IMAGES if k in p]
        chunks.append({
            "text": restored,
            "masked_text": p,
            "metadata": {
                "index": i,
                "tokens": len(enc.encode(restored)),
                "images": img_refs,
                "has_formula": any(k in p for k in FORMULAS),
                "has_table": any(k in p for k in TABLES),
                "has_image": bool(img_refs),
                "n_atomic": len(PH_RE.findall(p)),
            },
        })
    return chunks

In [128]:
text = Path(INPUT_MD).read_text(encoding="utf-8")
chunks = chunk_markdown(text)

Path(OUTPUT_JSON).write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

121204

In [129]:
sizes = [c["metadata"]["tokens"] for c in chunks]
big = [c for c in chunks if c["metadata"]["tokens"] > 1024]
print(f"Кол-во чанков: {len(chunks)}")
print(f"Размеры: min={min(sizes)}, avg={sum(sizes)//len(sizes)}, max={max(sizes)}")
print(f"Кол-во формул: {len(FORMULAS)} | Кол-во таблиц: {len(TABLES)} | Кол-во картинок: {len(IMAGES)}")
print(f"Чанков с формулой: {sum(c['metadata']['has_formula'] for c in chunks)}, "
      f"с таблицей: {sum(c['metadata']['has_table'] for c in chunks)}, "
      f"с картинкой: {sum(c['metadata']['has_image'] for c in chunks)}")
print(f"Чанков, превышающих лимит ({1024}): {len(big)}")

Кол-во чанков: 32
Размеры: min=620, avg=850, max=1867
Кол-во формул: 19 | Кол-во таблиц: 5 | Кол-во картинок: 26
Чанков с формулой: 9, с таблицей: 5, с картинкой: 12
Чанков, превышающих лимит (1024): 1


### Очистка
- Непечатные и управляющие символы (ASCII < 32, кроме \n, \r, \t)
- Лишние пробелы и табуляции
- HTML/XML-теги
- Стоп-слова (с осторожностью для технических текстов)
- Персональные данные (GDPR, ФЗ-152)
- Удаление картинок

In [137]:
import re

CTRL_RE      = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
INVISIBLE_RE = re.compile(r"[\u200b-\u200f\u2028-\u202f\u2060\ufeff\u00ad]")
SPACEY_RE    = re.compile(r"[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")
MULTISPACE_RE = re.compile(r"[ ]{2,}")
SPACES_NL_RE = re.compile(r"[ \t]+\n|\n[ \t]+")
SPACES_NL_CLEAN_RE = re.compile(r"[ \t]*\n[ \t]*")
MULTINL_RE    = re.compile(r"\n{3,}")
CR_RE         = re.compile(r"\r")
IMG_MASK = re.compile(r"\[\[IMAGE_\d{4}\]\]")

PII_PATTERNS = [
    re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),                              # email
    re.compile(r"\b(?:\+7|8)[\s\-\(]?\d{3}[\s\-\)]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}\b"),  # ru phone
]


def mask_pii(text):
    for pat in PII_PATTERNS:
        text = pat.sub("[PII]", text)
    return text

def count_issues(text):
    ctrl_hits       = CTRL_RE.findall(text)
    invisible_hits  = INVISIBLE_RE.findall(text)
    nbsp_hits       = SPACEY_RE.findall(text)
    multispace_hits = MULTISPACE_RE.findall(text)
    spaces_nl_hits  = SPACES_NL_RE.findall(text)
    multinl_hits    = MULTINL_RE.findall(text)
    pii_pattern1 = PII_PATTERNS[0].findall(text)
    pii_pattern2 = PII_PATTERNS[1].findall(text)

    return {
        "Управляющие символы ASCII": len(ctrl_hits),
        "Невидимые юникод-символы": len(invisible_hits),
        "Юникод-пробелы": len(nbsp_hits),
        "2+ подряд идущих пробелов": len(multispace_hits),
        "Пробелы перед/после перевода строки (висячие пробелы)": len(spaces_nl_hits),
        "3+ подряд идущих перевода строки": len(multinl_hits),
        "ФЗ-152": len(pii_pattern1) + len(pii_pattern2),
    }


def clean_chunk(text):
    text = CTRL_RE.sub("", text)
    text = INVISIBLE_RE.sub("", text)
    text = SPACEY_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text)
    text = SPACES_NL_CLEAN_RE.sub("\n", text)
    text = MULTINL_RE.sub("\n\n", text)
    text = IMG_MASK.sub("", text)
    for pat in PII_PATTERNS:
        text = pat.sub(" ", text)
    return text.strip()

def clean(chunks):
    return [clean_chunk(c["masked_text"]) for c in chunks]

In [138]:
for k, v in count_issues("".join([c["masked_text"] for c in chunks])).items():
    print(f"  {k:12}: {v}")
print()

  Управляющие символы ASCII: 0
  Невидимые юникод-символы: 0
  Юникод-пробелы: 0
  2+ подряд идущих пробелов: 14
  Пробелы перед/после перевода строки (висячие пробелы): 0
  3+ подряд идущих перевода строки: 7
  ФЗ-152      : 0



In [139]:
clean_chunks = clean(chunks) # has ALL masks

In [140]:
for k, v in count_issues("".join([c for c in clean_chunks])).items():
    print(f"  {k:12}: {v}")
print()

  Управляющие символы ASCII: 0
  Невидимые юникод-символы: 0
  Юникод-пробелы: 0
  2+ подряд идущих пробелов: 0
  Пробелы перед/после перевода строки (висячие пробелы): 0
  3+ подряд идущих перевода строки: 18
  ФЗ-152      : 0



In [136]:
from bert_score import score

P, R, F1 = score(
    cands=["\n".join([c["masked_text"] for c in chunks])],
    refs=["\n".join(clean_chunks)],
    lang="ru",
    verbose=False,
)
print(f"BERTScore F1 = {F1.item():.4f}")
print(f"BERTScore Precision = {P.item():.4f}")
print(f"BERTScore Recall = {R.item():.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14223.31it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore F1 = 1.0000
BERTScore Precision = 1.0000
BERTScore Recall = 1.0000


In [122]:
# Final: restore tables and formulas
clean_restored_chunks = [restore(clean_chunk) for clean_chunk in clean_chunks]